# SchoolBridge Fine-Tuning with Unsloth

Fine-tune Gemma 4 E4B for school notice extraction using QLoRA.

**Requirements**: Run on Kaggle with GPU T4 accelerator enabled.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-4b-it",
    max_seq_length=4096,
    load_in_4bit=True,
    full_finetuning=False,
)

model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Trainable parameters: {model.print_trainable_parameters()}")

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

In [ ]:
from datasets import load_dataset

# Upload train.jsonl and val.jsonl to your Kaggle dataset or use the path below
dataset = load_dataset("json", data_files={
    "train": "/kaggle/input/schoolbridge-training-data/train.jsonl",
    "validation": "/kaggle/input/schoolbridge-training-data/val.jsonl",
})

dataset = standardize_data_formats(dataset)

print(f"Train: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"\nSample:\n{dataset['train'][0]['conversations'][:2]}")

In [ ]:
from unsloth.chat_templates import train_on_responses_only

def apply_template(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True)

print("Sample formatted text (truncated):")
print(dataset["train"][0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        seed=42,
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=4096,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

In [ ]:
print("Starting training...")
stats = trainer.train()
print(f"\nTraining complete!")
print(f"Train loss: {stats.training_loss:.4f}")
print(f"Train runtime: {stats.metrics['train_runtime']:.0f}s")

In [ ]:
# Quick test inference
test_notice = """Dear Parent, Your child has 8 absences this semester exceeding the 5 absence threshold. 
Please contact the attendance office within 5 days. — Lincoln Elementary"""

messages = [
    {"role": "system", "content": "You are SchoolBridge, a school notice analyzer. Extract structured information and respond in valid JSON only."},
    {"role": "user", "content": f"Analyze this school notice:\n\n{test_notice}\n\nTarget language: English"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(inputs, max_new_tokens=1024, temperature=0.1, do_sample=True)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Model output:")
print(response)

In [ ]:
# Save and export to GGUF for Ollama
model.save_pretrained_gguf(
    "schoolbridge-gemma4-e4b",
    tokenizer,
    quantization_method="q4_k_m",
)

print("GGUF export complete! File: schoolbridge-gemma4-e4b/")
print("\nNext steps:")
print("1. Download the .gguf file from the output")
print("2. Create an Ollama Modelfile pointing to it")
print("3. Run: ollama create schoolbridge -f Modelfile")

In [ ]:
# Optional: push to Hugging Face Hub
# model.push_to_hub_gguf(
#     "YOUR_HF_USERNAME/schoolbridge-gemma4-e4b-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token="YOUR_HF_TOKEN",
# )